# Momentum Strategy Backtest — S&P 500
Testing a simple momentum strategy: buy when 20-day return > 2%, sell when it drops below -2%.
Period: 2019-2024

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Download data
df = yf.download('SPY', start='2019-01-01', end='2024-12-31', progress=False)
df['returns'] = df['Close'].pct_change()
df['mom_20'] = df['Close'].pct_change(20)
print(f'Data: {len(df)} bars, {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# Generate signals
ENTRY_THRESHOLD = 0.02
EXIT_THRESHOLD = -0.02

df['signal'] = 0
position = 0

for i in range(20, len(df)):
    if df['mom_20'].iloc[i] > ENTRY_THRESHOLD and position == 0:
        df.iloc[i, df.columns.get_loc('signal')] = 1
        position = 1
    elif df['mom_20'].iloc[i] < EXIT_THRESHOLD and position == 1:
        df.iloc[i, df.columns.get_loc('signal')] = -1
        position = 0

print(f'Buy signals: {(df["signal"] == 1).sum()}')
print(f'Sell signals: {(df["signal"] == -1).sum()}')

In [ ]:
# Backtest
capital = 100_000
cash = capital
shares = 0
equity = []

for i in range(len(df)):
    price = df['Close'].iloc[i]
    if df['signal'].iloc[i] == 1:
        shares = int(cash * 0.95 / price)
        cash -= shares * price
    elif df['signal'].iloc[i] == -1 and shares > 0:
        cash += shares * price
        shares = 0
    equity.append(cash + shares * price)

df['strategy_equity'] = equity
df['bh_equity'] = capital * (df['Close'] / df['Close'].iloc[0])

In [ ]:
# Plot equity curves
fig, ax = plt.subplots()
ax.plot(df.index, df['strategy_equity'], label='Momentum', color='#22d3ee', linewidth=1.5)
ax.plot(df.index, df['bh_equity'], label='Buy & Hold', color='#71717a', linewidth=1, linestyle='--')
ax.set_title('Momentum Strategy vs Buy & Hold (SPY)', fontsize=14)
ax.legend()
ax.set_ylabel('Portfolio Value ($)')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Performance metrics
strat_returns = df['strategy_equity'].pct_change().dropna()
total_ret = df['strategy_equity'].iloc[-1] / capital - 1
sharpe = strat_returns.mean() / strat_returns.std() * np.sqrt(252)
cum = (1 + strat_returns).cumprod()
max_dd = ((cum - cum.cummax()) / cum.cummax()).min()

print(f'Total Return:  {total_ret:.1%}')
print(f'Sharpe Ratio:  {sharpe:.2f}')
print(f'Max Drawdown:  {max_dd:.1%}')
print(f'B&H Return:    {df["bh_equity"].iloc[-1] / capital - 1:.1%}')